<!-- track-identity-card -->
# Cluster separation as coverage widens

| | |
|---|---|
| Pipeline step | `06_tier_ladder.ipynb` |
| Manuscript section | 4.3 |
| Copied from | `notebooks/NB11_tier_pipeline_v3_2_2026-07-22_2100.ipynb` |
| Source sha256 | `10de859fef65c68b191ef62e63e81a9c` |

**Reads**

- `data/processed/<tier>/*_human.parquet`
- `data/fingerprints/t<n>_<policy>/t<n>_info.json`

**Writes**

- `data/fingerprints/t<n>_<policy>/t<n>_info.json`

Produces the tier ladder of Section 4.3, each coefficient against its own permutation null over 1000 permutations.

> Copied from the working notebook named above. Two changes were made to it: this identity card and the bootstrap cell that follows it were added, and the hard-coded data paths were replaced with the root that the bootstrap cell resolves. The analysis code is unchanged.


# NB11 — Tier Pipeline Wrapper v2

**Amac:** NB08 v4 + NB09 v3 + NB10A + NB10B pipeline'ini T1/T2/T3/T4 tier'lariyla calistirir.

**v1'den farklar (yalnizca dort nokta — analiz mantigi degismedi):**

1. **T1 artik bir tier.** `TIER_DIRS` ve `TIER_CARS` sozluklerine T1 (yalnizca `bmw_z4_gt3`) eklendi.
   v1'de T1 calistirilamiyordu, degerleri elle yazilmisti.
2. **T1 referans degerleri hardcode degil.** v1'de uc yerde `LOOCV=62.5%`, `rho=0.984`,
   `#1=coast_dist` ve `t1_vals = {...}` sabitleri vardi. Hepsi `t1_info.json` okumasiyla degistirildi.
3. **T1 ciktisi kendi dizinine yaziyor.** `T1_FP_DIR` artik `fingerprints/` koku degil `fingerprints/t1`.
   Koke NB09 v3 yaziyor; ikisinin karismasi v1'deki temel belirsizligin kaynagiydi.
4. **`n_drivers < 4` korumasi.** v1'de bu dalda `acc`/`rho`/`top5` tanimsiz kalip ozet hucresi
   `NameError` veriyordu. Sentinel deger atanip ozet hucresi korundu.
5. **Isim cakismasi giderildi.** Faz 1'in son hucresinde pist-basi surucu sayisi v1'de
   `n_drivers` adiyla tutuluyordu; Faz 3 ayni adi cross-track sayisiyla eziyor. Faz 1'deki
   degisken `n_drivers_t` olarak yeniden adlandirildi. Sirali calistirmada v1 de dogru
   sonuc verir; fark yalnizca Faz 3 atlandiginda ortaya cikar.
6. **Bos-veri korumasi (Faz 2).** Hicbir pist yuklenemediginde v1
   `set.intersection()` cagrisinda anlamsiz bir `TypeError` veriyordu. Artik asil nedeni
   soyleyen bir uyari basiliyor. Ayrica pist basina surucu sayisi kesisimden once
   ayri ayri raporlaniyor — kesisimin neden kucuk kaldigi gorulebilsin diye.

**Ek olarak** `info` sozlugune `n_drivers_per_track` ve `degenerate_ids_excluded` alanlari eklendi
(paper_numbers icin kaynak izlenebilirligi).

**v1 arsiv olarak korunur** — T3/T4 calistirmasinin kayit defteri odur, uzerine yazilmaz.

---
**Kullanim:** Asagidaki hucrede `TIER` degerini degistir, notebook'u bastan sona calistir.
Sira onemli: once `TIER="T1"`, sonra digerleri (T2/T3/T4 baseline karsilastirmasi icin
`t1_info.json` dosyasina ihtiyac duyar).

## v3 — kosu sirasi

Toplam **8 kosu**. Once `POLICY = "P_INC"` ile T1->T2->T3->T4, sonra `POLICY = "P_EXC"` ile ayni sira.
Cikti dizinleri `_inc`/`_exc` eki tasir; iki kosu birbirini ezmez, eski `fingerprints/t1..t4` arsivi dokunulmadan kalir.

**v3:** (1) ortak kimlik politikasi, `20240501_MPC` ilk kez dislaniyor. (2) politika ekli cikti dizinleri. (3) `*_info.json` artik `identity_policy`, `ablock_excluded`, `exclude_hard`, `loocv_n_correct/total`, `cross_track_driver_ids` yaziyor — sekiz JSON tek basina yeterli.


In [ ]:
# track-config-bootstrap
# Locates track/config.py, which resolves the data root at run time.
# Works from a flat layout (track/ beside the notebooks) and from the
# repository layout (src/track/ one level up). See track/config.py.
import sys as _sys, pathlib as _pl
_cands = []
for _p in [_pl.Path.cwd()] + list(_pl.Path.cwd().parents):
    _cands += [_p, _p / "src"]
for _c in _cands:
    if (_c / "track" / "config.py").is_file():
        _sys.path.insert(0, str(_c))
        break
else:
    raise RuntimeError(
        "track/config.py not found. Run this notebook from inside the repository, "
        "or add the directory holding track/ to sys.path."
    )
from track.config import PROJECT_ROOT as TRACK_ROOT
print("data root:", TRACK_ROOT)


In [ ]:
# ╔══════════════════════════════════════════════════╗
# ║  KOSU AYARLARI — SADECE BURAYI DEGISTIR          ║
# ╚══════════════════════════════════════════════════╝
TIER   = "T1"      # "T1", "T2", "T3", veya "T4"
POLICY = "P_INC"   # "P_INC" = A-blogu DAHIL | "P_EXC" = A-blogu HARIC

# ══════════════════════════════════════════════════
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.signal import savgol_filter
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import LeaveOneOut, cross_val_predict
from sklearn.metrics import accuracy_score
from scipy.stats import spearmanr
from collections import defaultdict
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

plt.style.use('dark_background')

PROJECT      = TRACK_ROOT
FEATURES_DIR = PROJECT / "data" / "features"
FP_DIR       = PROJECT / "data" / "fingerprints"
FIG_DIR      = PROJECT / "results" / "figures"

# --- v3 DEGISIKLIK 1: ORTAK KIMLIK POLITIKASI (tek kaynak) ---
EXCLUDE_HARD = [
    "20240213_000000_ACAI",   # oyun-ici yapay surucu
    "20240308_ensemble",      # RL politika ciktisi
    "20240501_MPC",           # klasik kontrol baseline (veri kumesi belgesi)
]
ABLOCK = [
    "20240410_A_12_123",
    "20240410_A_21_231",
    "20240411_A_12_312",
]
assert POLICY in ("P_INC","P_EXC"), "POLICY 'P_INC' ya da 'P_EXC' olmali"
POL_SUF = "_inc" if POLICY == "P_INC" else "_exc"

# --- v2 DEGISIKLIK 3 ---
# v1: T1_FP_DIR = FP_DIR  (fingerprints/ koku = NB09 v3'un yazdigi yer)
# v2: T1 kendi tier dizinine yazar; NB09 v3 ciktisiyla karismaz.
# v3: politika eki — iki politika birbirinin t1_info.json'ini EZMEZ.
T1_FP_DIR    = FP_DIR / ("t1" + POL_SUF)


def load_baseline_t1():
    """T1 referans degerlerini t1_info.json'dan okur. Dosya yoksa None doner.

    v1'de silhouette / LOOCV / rho / top1 degerleri uc ayri yerde elle yaziliydi.
    Tek kaynak buradan okunur; hicbir yerde tekrar yazilmaz.
    """
    import json as _json
    p = T1_FP_DIR / "t1_info.json"
    if not p.exists():
        return None
    with open(p, encoding="utf-8") as f:
        return _json.load(f)


# --- v2 DEGISIKLIK 1: T1 eklendi ---
TIER_DIRS = {
    "T1": PROJECT / "data" / "processed" / "tier1_small_3t1c",
    "T2": PROJECT / "data" / "processed" / "tier2_medium_3t2c",
    "T3": PROJECT / "data" / "processed" / "tier3_large_3t3c",
    "T4": PROJECT / "data" / "processed" / "tier4_full_4t3c",
}

TIER_CARS = {
    "T1": ["bmw_z4_gt3"],
    "T2": ["bmw_z4_gt3", "dallara_f317"],
    "T3": ["bmw_z4_gt3", "dallara_f317", "ks_mazda_miata"],
    "T4": ["bmw_z4_gt3", "dallara_f317", "ks_mazda_miata"],
}

TRACKS = ['monza', 'barcelona', 'red_bull_ring']

# Tier-specific output directories
# v3 DEGISIKLIK 2: cikti dizinleri politika eki tasir (eski arsiv korunur)
TIER_FEATURES = FEATURES_DIR / (TIER.lower() + POL_SUF)
TIER_FP       = FP_DIR / (TIER.lower() + POL_SUF)
TIER_FIG      = FIG_DIR / (TIER.lower() + POL_SUF)
for d in [TIER_FEATURES, TIER_FP, TIER_FIG]:
    d.mkdir(parents=True, exist_ok=True)

TIER_DIR = TIER_DIRS[TIER]
TIER_CAR_LIST = TIER_CARS[TIER]

print(f"Secilen tier: {TIER}   |   Kimlik politikasi: {POLICY}")
print(f"Dizin: {TIER_DIR}")
print(f"Dizin var mi: {TIER_DIR.exists()}")
print(f"Araclar: {TIER_CAR_LIST}")
print(f"Cikti: features/{TIER.lower()+POL_SUF}, fingerprints/{TIER.lower()+POL_SUF}, figures/{TIER.lower()+POL_SUF}")

if not TIER_DIR.exists():
    print()
    print("!!! DIZIN BULUNAMADI — TIER_DIRS icindeki yolu duzelt.")
    print("    Mevcut processed alt dizinleri:")
    _proc = PROJECT / "data" / "processed"
    if _proc.exists():
        for _d in sorted(_proc.iterdir()):
            if _d.is_dir():
                _n = len(list(_d.glob("*_human.parquet")))
                print(f"      {_d.name:<30s} {_n:>3d} *_human.parquet")

## Faz 0: Tier Dosya Kesfetme

In [ ]:
# Sadece aggregated dosyalari bul (*_human.parquet)
all_files = sorted(TIER_DIR.glob("*_human.parquet"))
print(f"{TIER} aggregated dosya sayisi: {len(all_files)}")

# Track x Car matrisi
file_map = {}
for f in all_files:
    name = f.stem.replace("_human", "")
    # Track ve car'i ayir
    for track in TRACKS:
        if name.startswith(track + "_"):
            car = name[len(track) + 1:]
            file_map[(track, car)] = f
            break

print(f"\nTrack x Car matrisi:")
cars_found = sorted(set(c for _, c in file_map.keys()))
print(f"  Bulunan araclar: {cars_found}")
print(f"  {'':>20s}", end="")
for car in cars_found:
    print(f"  {car[:12]:>12s}", end="")
print()

for track in TRACKS:
    print(f"  {track:>20s}", end="")
    for car in cars_found:
        if (track, car) in file_map:
            size = file_map[(track, car)].stat().st_size / 1e6
            print(f"  {size:>9.1f} MB", end="")
        else:
            print(f"  {'---':>12s}", end="")
    print()

# Filtreleme: sadece TIER_CAR_LIST'teki araclari al
tier_files = {k: v for k, v in file_map.items() if k[1] in TIER_CAR_LIST}
excluded = {k: v for k, v in file_map.items() if k[1] not in TIER_CAR_LIST}
print(f"\nTier'a dahil: {len(tier_files)} dosya")
if excluded:
    print(f"Haric birakildi: {[f'{t}_{c}' for (t,c) in excluded.keys()]}")

# --- v2 EK KONTROL ---
# T1'in gercekten tek-araclik oldugunu dogrula. v1'de bu kontrol yoktu ve
# T1'in hangi araclari icerdigi hicbir yerde kayitli degildi.
_cars_in_tier = sorted(set(c for _, c in tier_files.keys()))
print(f"Tier'da fiilen bulunan araclar: {_cars_in_tier}")
if set(_cars_in_tier) != set(TIER_CAR_LIST):
    print(f"  UYARI: beklenen {TIER_CAR_LIST}, bulunan {_cars_in_tier}")

In [ ]:
# Telemetri yukle — track bazinda birlestir (tum araclar)
loaded = {}

# Degenerate surucu filtresi — TEK TANIM (v1'de dongu icindeydi, disari alindi
# ki info JSON'a yazilabilsin ve makaledeki dislama kuralinin kaynagi tek yerde olsun)
# v3: tek kaynak — EXCLUDE_HARD + (politikaya gore) ABLOCK
DEGENERATE_IDS = EXCLUDE_HARD + ([] if POLICY == "P_INC" else ABLOCK)
print(f"    Kimlik politikasi: {POLICY} -> {len(DEGENERATE_IDS)} kimlik dislaniyor")

for track in TRACKS:
    track_files = [(car, path) for (t, car), path in tier_files.items() if t == track]
    if not track_files:
        print(f"  {track}: DOSYA YOK — atlaniyor")
        continue

    # Corners (track geometrisi — car-independent)
    corners_path = FEATURES_DIR / f"{track}_corners_v3.parquet"
    if not corners_path.exists():
        print(f"  {track}: corners_v3 BULUNAMADI — atlaniyor")
        continue

    corners_v3 = pd.read_parquet(corners_path)

    # Telemetriyi birlestir
    dfs = []
    for car, path in track_files:
        df = pd.read_parquet(path)
        if 'car' not in df.columns:
            df['car'] = car
        dfs.append(df)
        print(f"  {track}/{car}: {len(df):,} satir, "
              f"{df['driver_id'].nunique() if 'driver_id' in df.columns else '?'} surucu")

    tele_all = pd.concat(dfs, ignore_index=True)

    # Degenerate surucu filtresi (ACAI, ensemble)
    if 'driver_id' in tele_all.columns:
        n_before = tele_all['driver_id'].nunique()
        tele_all = tele_all[~tele_all['driver_id'].isin(DEGENERATE_IDS)]
        n_after = tele_all['driver_id'].nunique()
        if n_before != n_after:
            print(f"    Degenerate filtre: {n_before} -> {n_after} surucu")

    loaded[track] = {
        'corners_v3': corners_v3,
        'tele_all': tele_all,
    }

print(f"\nYuklenen pistler: {list(loaded.keys())}")
total_drivers = set()
for data in loaded.values():
    if 'driver_id' in data['tele_all'].columns:
        total_drivers.update(data['tele_all']['driver_id'].unique())
print(f"Toplam benzersiz surucu: {len(total_drivers)}")

# --- v2 EK RAPOR: pist basina surucu sayisi acikca yazilsin ---
print(f"\nPist basina surucu (degenerate filtresi sonrasi):")
for track, data in loaded.items():
    if 'driver_id' in data['tele_all'].columns:
        print(f"  {track:<16s} {data['tele_all']['driver_id'].nunique():>3d}")

## Faz 1: NB08 v4 — Corner Segmentation

In [ ]:
# ══════════════════════════════════════════════════
# NB08 v4 CORE FUNCTIONS (aynen kopyalandi)
# ══════════════════════════════════════════════════

# Parametreler
SEARCH_BACK_M      = 300
SEARCH_FWD_M       = 200
TRAIL_BRAKE_ZONE   = 0.40
TRAIL_MIN_PRESSURE = 0.03
MIN_APEX_SPEED     = 20.0
MIN_ENTRY_SPEED    = 50.0
MC_RADIUS_M        = 15
EXIT_FALLBACK_M    = 50

def detect_steer_column(df):
    cols = {c: c.lower() for c in df.columns}
    for c, cl in cols.items():
        if any(k in cl for k in ['steerangle', 'steering_angle', 'steer_angle', 'wheel_angle']):
            return ('steer', c)
    for c, cl in cols.items():
        if 'steer' in cl and 'error' not in cl:
            return ('steer', c)
    for c, cl in cols.items():
        if any(k in cl for k in ['g_lat', 'glat', 'lateral_g', 'accg_y', 'accel_lat']):
            return ('g_lat', c)
    return ('speed_proxy', None)

def votes_to_confidence(n_votes):
    if n_votes >= 3: return 1.0
    elif n_votes == 2: return 0.7
    elif n_votes == 1: return 0.4
    return 0.0

def _empty_seg(apex_dist, apex_speed):
    return {
        'seg_apex_dist': apex_dist, 'seg_apex_speed': apex_speed,
        'seg_entry_dist': np.nan, 'seg_entry_speed': np.nan,
        'seg_exit_dist': np.nan, 'seg_exit_speed': np.nan,
        'seg_braking_dist': np.nan, 'seg_speed_loss_eff': np.nan,
        'seg_coasting_dist': np.nan, 'seg_trail_braking': False,
        'seg_trail_pressure': np.nan, 'seg_avg_brake_pressure': np.nan,
        'seg_entry_valid': False, 'seg_exit_method': 'none',
    }

# Steer column tespit
for track_name, data in loaded.items():
    mode, col = detect_steer_column(data['tele_all'])
    data['steer_info'] = (mode, col)
    print(f"  {track_name}: mode={mode}, column={col}")

In [ ]:
def segment_corner(df_lap, apex_lapdist, difficulty):
    speed = df_lap['speed_kmh'].values
    brake = df_lap['brakeStatus'].values if 'brakeStatus' in df_lap.columns else np.zeros(len(df_lap))
    acc   = df_lap['accStatus'].values   if 'accStatus'   in df_lap.columns else np.zeros(len(df_lap))
    dist  = df_lap['LapDist'].values

    apex_idx = int(np.argmin(np.abs(dist - apex_lapdist)))
    apex_speed = float(speed[apex_idx])

    # Entry
    back_limit = max(0, apex_idx - int(SEARCH_BACK_M / 2))
    entry_idx  = apex_idx
    for j in range(apex_idx - 1, back_limit, -1):
        if brake[j] < 0.05 and speed[j] > speed[apex_idx]:
            entry_idx = j
            break

    entry_speed = float(speed[entry_idx])
    if entry_speed < MIN_ENTRY_SPEED or apex_speed < MIN_APEX_SPEED:
        return _empty_seg(apex_lapdist, apex_speed)

    # EXIT v4: 3-katmanli tespit
    fwd_limit = min(len(speed) - 1, apex_idx + int(SEARCH_FWD_M / 2))
    exit_idx    = apex_idx
    exit_method = 'none'

    for j in range(apex_idx + 1, fwd_limit):
        if acc[j] > 0.3 and speed[j] > apex_speed:
            exit_idx = j; exit_method = 'throttle_full'; break

    if exit_idx == apex_idx:
        for j in range(apex_idx + 1, fwd_limit):
            if acc[j] > 0.3:
                exit_idx = j; exit_method = 'throttle_only'; break

    if exit_idx == apex_idx:
        fb_target = dist[apex_idx] + EXIT_FALLBACK_M
        fb_idx = int(np.argmin(np.abs(dist - fb_target)))
        exit_idx = min(fb_idx, fwd_limit)
        exit_method = 'distance_fallback'

    exit_speed = float(speed[exit_idx])

    entry_dist_v = float(dist[entry_idx])
    exit_dist_v  = float(dist[exit_idx])
    braking_dist = float(dist[apex_idx] - dist[entry_idx]) if entry_idx < apex_idx else 0.0
    speed_loss_eff = (apex_speed / entry_speed) if entry_speed > 0 else 0.0

    # Coasting
    coast_start = apex_idx
    for j in range(apex_idx, fwd_limit):
        if brake[j] < 0.05:
            coast_start = j; break
    coast_end = coast_start
    for j in range(coast_start, fwd_limit):
        if acc[j] > 0.1:
            coast_end = j; break
    coasting_dist = float(dist[coast_end] - dist[coast_start]) if coast_end > coast_start else 0.0

    # Trail braking
    trail_braking = False
    trail_pressure = 0.0
    avg_brake = 0.0
    if braking_dist > 10:
        trail_zone_start = entry_idx + int((apex_idx - entry_idx) * (1 - TRAIL_BRAKE_ZONE))
        trail_pressures  = brake[trail_zone_start:apex_idx]
        trail_braking    = bool(trail_pressures.mean() > TRAIL_MIN_PRESSURE) if len(trail_pressures) > 0 else False
        trail_pressure   = float(trail_pressures.mean()) if len(trail_pressures) > 0 else 0.0
        brake_zone       = brake[entry_idx:apex_idx]
        avg_brake        = float(brake_zone.mean()) if len(brake_zone) > 0 else 0.0

    return {
        'seg_apex_dist': apex_lapdist, 'seg_apex_speed': apex_speed,
        'seg_entry_dist': entry_dist_v, 'seg_entry_speed': entry_speed,
        'seg_exit_dist': exit_dist_v, 'seg_exit_speed': exit_speed,
        'seg_braking_dist': braking_dist, 'seg_speed_loss_eff': speed_loss_eff,
        'seg_coasting_dist': coasting_dist, 'seg_trail_braking': trail_braking,
        'seg_trail_pressure': trail_pressure, 'seg_avg_brake_pressure': avg_brake,
        'seg_entry_valid': True, 'seg_exit_method': exit_method,
    }

print("segment_corner() v4 hazir")

In [ ]:
def segment_corner_phases(df_lap, seg, steer_info):
    mode, steer_col = steer_info

    if not seg.get('seg_entry_valid', False) or pd.isna(seg.get('seg_entry_dist')):
        return {
            'phase_slb_dist': np.nan, 'phase_slb_decel_rate': np.nan,
            'phase_ce_dist': np.nan, 'phase_ce_brake_at_turnin': np.nan,
            'phase_mc_speed_ratio': np.nan, 'phase_mc_lateral_signal': np.nan,
            'phase_cex_throttle_lag': np.nan, 'phase_cex_accel_rate': np.nan,
            'phase_turnin_method': 'skipped',
        }

    dist  = df_lap['LapDist'].values
    speed = df_lap['speed_kmh'].values
    brake = df_lap['brakeStatus'].values if 'brakeStatus' in df_lap.columns else np.zeros(len(df_lap))
    acc   = df_lap['accStatus'].values if 'accStatus' in df_lap.columns else np.zeros(len(df_lap))

    entry_dist = seg['seg_entry_dist']
    apex_dist  = seg['seg_apex_dist']
    exit_dist  = seg['seg_exit_dist']

    entry_idx = int(np.argmin(np.abs(dist - entry_dist)))
    apex_idx  = int(np.argmin(np.abs(dist - apex_dist)))
    exit_idx  = int(np.argmin(np.abs(dist - exit_dist)))

    # Turn-in noktasi
    turnin_idx = entry_idx + int((apex_idx - entry_idx) * 0.4)
    turnin_method = 'speed_proxy_40pct'

    if mode == 'steer' and steer_col in df_lap.columns:
        steer = np.abs(df_lap[steer_col].values)
        region = steer[entry_idx:apex_idx]
        if len(region) > 5:
            wlen = min(11, len(region))
            if wlen % 2 == 0: wlen -= 1
            if wlen >= 3:
                smoothed = savgol_filter(region, wlen, min(2, wlen - 1))
            else:
                smoothed = region
            threshold = smoothed.max() * 0.15
            for k, val in enumerate(smoothed):
                if val > threshold:
                    turnin_idx = entry_idx + k
                    turnin_method = 'steer_threshold'
                    break

    elif mode == 'g_lat' and steer_col in df_lap.columns:
        glat = np.abs(df_lap[steer_col].values)
        region = glat[entry_idx:apex_idx]
        if len(region) > 5:
            threshold = region.max() * 0.20
            for k, val in enumerate(region):
                if val > threshold:
                    turnin_idx = entry_idx + k
                    turnin_method = 'g_lat_threshold'
                    break

    turnin_idx = max(entry_idx + 1, min(turnin_idx, apex_idx - 1))

    # MC fazi
    mc_start = int(np.argmin(np.abs(dist - (apex_dist - MC_RADIUS_M))))
    mc_end   = int(np.argmin(np.abs(dist - (apex_dist + MC_RADIUS_M))))
    mc_start = max(mc_start, turnin_idx)
    mc_end   = min(mc_end, exit_idx)

    # SLB: entry -> turnin
    slb_dist = float(dist[turnin_idx] - dist[entry_idx]) if turnin_idx > entry_idx else 0.0
    slb_speed_drop = float(speed[entry_idx] - speed[turnin_idx])
    slb_decel_rate = slb_speed_drop / slb_dist if slb_dist > 5 else 0.0

    # CE: turnin -> mc_start
    ce_dist = float(dist[mc_start] - dist[turnin_idx]) if mc_start > turnin_idx else 0.0
    ce_brake_at_turnin = float(brake[turnin_idx]) if turnin_idx < len(brake) else 0.0

    # MC
    mc_speed = float(speed[mc_start:mc_end+1].mean()) if mc_end > mc_start else float(speed[apex_idx])
    entry_speed_val = float(speed[entry_idx])
    mc_speed_ratio = mc_speed / entry_speed_val if entry_speed_val > 0 else 0.0

    mc_lateral = 0.0
    if mode in ('steer', 'g_lat') and steer_col and steer_col in df_lap.columns:
        lat_vals = np.abs(df_lap[steer_col].values[mc_start:mc_end+1])
        mc_lateral = float(lat_vals.mean()) if len(lat_vals) > 0 else 0.0

    # CEX: mc_end -> exit
    cex_throttle_lag = 0.0
    cex_accel_rate = 0.0
    if exit_idx > mc_end:
        for j in range(mc_end, exit_idx):
            if acc[j] > 0.1:
                cex_throttle_lag = float(dist[j] - dist[mc_end])
                break
        speed_gain = float(speed[exit_idx] - speed[mc_end])
        cex_dist = float(dist[exit_idx] - dist[mc_end])
        cex_accel_rate = speed_gain / cex_dist if cex_dist > 5 else 0.0

    return {
        'phase_slb_dist': slb_dist, 'phase_slb_decel_rate': slb_decel_rate,
        'phase_ce_dist': ce_dist, 'phase_ce_brake_at_turnin': ce_brake_at_turnin,
        'phase_mc_speed_ratio': mc_speed_ratio, 'phase_mc_lateral_signal': mc_lateral,
        'phase_cex_throttle_lag': cex_throttle_lag, 'phase_cex_accel_rate': cex_accel_rate,
        'phase_turnin_method': turnin_method,
    }

print("segment_corner_phases() hazir")

In [ ]:
def enrich_corners(corners_v3_df, df_single_driver, steer_info, track_name=None):
    segs = []
    for _, row in corners_v3_df.iterrows():
        seg = segment_corner(
            df_lap       = df_single_driver,
            apex_lapdist = float(row['apex_dist']),
            difficulty   = float(row.get('difficulty_score', 0.5)),
        )
        phase = segment_corner_phases(df_single_driver, seg, steer_info)
        seg.update(phase)
        segs.append(seg)

    enriched = corners_v3_df.reset_index(drop=True).join(pd.DataFrame(segs))
    enriched['confidence'] = enriched['n_votes'].apply(votes_to_confidence)
    enriched.loc[enriched['seg_entry_dist'].isna(), 'confidence'] = 0.0

    bad_apex = enriched['seg_apex_speed'] < MIN_APEX_SPEED
    if bad_apex.any():
        enriched.loc[bad_apex, 'confidence'] = 0.0

    bad_entry = enriched['seg_entry_speed'] < enriched['seg_apex_speed']
    if bad_entry.any():
        enriched.loc[bad_entry, 'confidence'] = 0.0

    if track_name:
        ok = (enriched['confidence'] > 0).sum()
        print(f"  [{track_name}] {ok}/{len(enriched)} viraj gecerli")

    return enriched

print("enrich_corners() hazir")

In [ ]:
# Segmentasyon calistir
all_enriched = {}

for track_name, data in loaded.items():
    print(f"\n{'='*50}")
    print(f"  {track_name.upper()} — {TIER}")
    print(f"{'='*50}")

    corners_v3  = data['corners_v3']
    tele_all    = data['tele_all']
    steer_info  = data['steer_info']
    driver_ids  = sorted(tele_all['driver_id'].unique()) if 'driver_id' in tele_all.columns else ['all']

    track_results = []
    for driver_id in driver_ids:
        df_drv = tele_all[tele_all['driver_id'] == driver_id].reset_index(drop=True) \
                 if 'driver_id' in tele_all.columns else tele_all.copy()

        required = ['speed_kmh', 'LapDist', 'brakeStatus', 'accStatus']
        missing  = [c for c in required if c not in df_drv.columns]
        if missing:
            continue

        enriched = enrich_corners(corners_v3, df_drv, steer_info)
        enriched['driver_id'] = driver_id
        if 'car' in df_drv.columns:
            enriched['car'] = df_drv['car'].iloc[0]
        track_results.append(enriched)

    if track_results:
        all_enriched[track_name] = pd.concat(track_results, ignore_index=True)
        n_drivers_t = all_enriched[track_name]['driver_id'].nunique()
        n_valid = (all_enriched[track_name]['confidence'] > 0).sum()
        print(f"  Toplam: {n_drivers_t} surucu, {n_valid} gecerli viraj segmenti")

print(f"\nSegmentasyon tamamlandi: {list(all_enriched.keys())}")

## Faz 2: NB09 v3 — Driver Fingerprint

In [ ]:
def wavg(values, weights):
    v    = pd.to_numeric(pd.Series(values), errors='coerce').values
    w    = np.array(weights, dtype=float)
    mask = ~np.isnan(v) & (w > 0)
    return float(np.average(v[mask], weights=w[mask])) if mask.sum() > 0 else np.nan

def build_driver_matrix(track_name):
    if track_name not in loaded or track_name not in all_enriched:
        return None

    corners_v3 = loaded[track_name]['corners_v3']
    tele_all   = loaded[track_name]['tele_all']
    steer_info = loaded[track_name]['steer_info']
    driver_ids = sorted(tele_all['driver_id'].unique()) if 'driver_id' in tele_all.columns else ['all']

    all_records = []

    for driver_id in driver_ids:
        df_drv = tele_all[tele_all['driver_id'] == driver_id].reset_index(drop=True) \
                 if 'driver_id' in tele_all.columns else tele_all.copy()

        required = ['speed_kmh', 'LapDist', 'brakeStatus', 'accStatus']
        if any(c not in df_drv.columns for c in required):
            continue

        enriched = enrich_corners(corners_v3, df_drv, steer_info)
        w     = enriched['confidence'].fillna(0).values
        valid = enriched[enriched['confidence'] > 0]

        if len(valid) == 0:
            continue

        record = {
            'driver_id': driver_id, 'track': track_name,
            'n_corners_total': len(enriched), 'n_corners_valid': len(valid),
            'mean_apex_speed':     wavg(enriched['seg_apex_speed'], w),
            'mean_entry_speed':    wavg(enriched['seg_entry_speed'], w),
            'mean_exit_speed':     wavg(enriched['seg_exit_speed'], w),
            'speed_loss_eff':      wavg(enriched['seg_speed_loss_eff'], w),
            'mean_braking_dist':   wavg(enriched['seg_braking_dist'], w),
            'mean_brake_pressure': wavg(enriched['seg_avg_brake_pressure'], w),
            'mean_coasting_dist':  wavg(enriched['seg_coasting_dist'], w),
            'trail_braking_ratio': float(valid['seg_trail_braking'].mean()),
            'mean_trail_pressure': wavg(enriched['seg_trail_pressure'], w),
            'apex_speed_std':      float(valid['seg_apex_speed'].std()) if len(valid) > 1 else np.nan,
            'braking_dist_std':    float(valid['seg_braking_dist'].std()) if len(valid) > 1 else np.nan,
            'exit_speed_std':      float(valid['seg_exit_speed'].std()) if len(valid) > 1 else np.nan,
            'speed_loss_eff_std':  float(valid['seg_speed_loss_eff'].std()) if len(valid) > 1 else np.nan,
        }

        # Character class metrikleri
        if 'character_class' in valid.columns:
            for cc in ['heavy_braking', 'trail_braking', 'lift_coast', 'flat_out']:
                record[f'pct_{cc}'] = float((valid['character_class'] == cc).mean())

        # Phase metrikleri
        phase_cols_map = {
            'mean_slb_dist': 'phase_slb_dist', 'mean_slb_decel': 'phase_slb_decel_rate',
            'mean_ce_brake_turnin': 'phase_ce_brake_at_turnin',
            'mean_mc_speed_ratio': 'phase_mc_speed_ratio', 'mean_mc_lateral': 'phase_mc_lateral_signal',
            'mean_cex_throttle_lag': 'phase_cex_throttle_lag', 'mean_cex_accel_rate': 'phase_cex_accel_rate',
        }
        for out_col, src_col in phase_cols_map.items():
            if src_col in enriched.columns:
                record[out_col] = wavg(enriched[src_col], w)

        all_records.append(record)

    if not all_records:
        return None

    mat = pd.DataFrame(all_records)
    return mat

# Build matrices
matrices = {}
for track in TRACKS:
    if track not in loaded:
        continue
    mat = build_driver_matrix(track)
    if mat is not None:
        matrices[track] = mat
        # Negatif clipping
        if 'mean_slb_dist' in mat.columns:
            neg = (mat['mean_slb_dist'] < 0).sum()
            if neg > 0:
                mat['mean_slb_dist'] = mat['mean_slb_dist'].clip(lower=0)
                print(f"  {track}: {neg} negatif slb_dist cliplendi")

        # Kaydet
        mat.to_parquet(TIER_FEATURES / f'driver_corner_matrix_{track}.parquet', index=False)
        print(f"  {track}: {len(mat)} surucu matrisi kaydedildi")

print(f"\nMatrisler hazir: {list(matrices.keys())}")

In [ ]:
# 19 metrikli boyut haritasi
DIMENSIONS = {
    'B1_Hiz': {
        'label': 'Hiz Yonetimi',
        'metrics': ['mean_apex_speed', 'speed_loss_eff', 'mean_mc_speed_ratio',
                     'mean_mc_lateral', 'mean_cex_accel_rate']
    },
    'B2_Frenleme': {
        'label': 'Frenleme Stili',
        'metrics': ['mean_brake_pressure', 'trail_braking_ratio', 'mean_trail_pressure',
                     'mean_slb_dist', 'mean_slb_decel', 'mean_ce_brake_turnin']
    },
    'B3_Strateji': {
        'label': 'Surus Stratejisi',
        'metrics': ['mean_coasting_dist', 'mean_cex_throttle_lag', 'pct_lift_coast', 'pct_flat_out']
    },
    'B4_Tutarlilik': {
        'label': 'Tutarlilik',
        'metrics': ['apex_speed_std', 'exit_speed_std', 'speed_loss_eff_std', 'braking_dist_std']
    },
}
DIM_NAMES = list(DIMENSIONS.keys())
ALL_METRICS = []
for cfg in DIMENSIONS.values():
    ALL_METRICS.extend(cfg['metrics'])

def compute_fingerprint_track(df_track, track_name):
    rows = []
    for _, drv in df_track.iterrows():
        row = {'driver_id': drv['driver_id'], 'track': track_name}
        for dim, cfg in DIMENSIONS.items():
            vals = [float(drv[m]) for m in cfg['metrics']
                    if m in drv.index and pd.notna(drv[m])]
            row[dim] = np.mean(vals) if vals else np.nan
        rows.append(row)
    fp = pd.DataFrame(rows)
    for dim in DIM_NAMES:
        col = fp[dim].copy()
        if dim == 'B4_Tutarlilik':
            col = 1.0 - col
        mn, mx = col.min(), col.max()
        if mx > mn:
            fp[dim] = (col - mn) / (mx - mn)
        else:
            fp[dim] = 0.5
    return fp

# Per-track fingerprint
fps_track = {}
for track in TRACKS:
    if track not in matrices:
        continue
    fps_track[track] = compute_fingerprint_track(matrices[track], track)
    fp = fps_track[track]
    scores = ' '.join(f"B{i+1}:{fp[d].mean():.3f}" for i, d in enumerate(DIM_NAMES))
    print(f"  [{track}] {len(fp)} surucu | {scores}")

# Cross-track
# --- v2 KORUMA ---
# v1'de fps_track bos oldugunda set.intersection() argumansiz cagriliyor ve
# "unbound method set.intersection() needs an argument" TypeError'i veriyordu.
# Asil neden genellikle yukarida: dosya bulunamamis ya da tier dizini yanlis.
if not fps_track:
    common = []
    fp_cross = pd.DataFrame(columns=['driver_id', 'n_tracks'] + DIM_NAMES)
    print("\n!!! Hicbir pist icin fingerprint uretilmedi.")
    print("    Yukaridaki Faz 0 ciktisini kontrol et: dosya bulundu mu, tier dizini dogru mu?")
else:
    driver_sets = {t: set(fps_track[t]['driver_id']) for t in fps_track}
    common = sorted(set.intersection(*driver_sets.values()))
    print(f"\nOrtak surucu sayisi (tum pistlerde): {len(common)}")
    for t, s in driver_sets.items():
        print(f"  {t:<16s} {len(s):>3d} surucu")

def compute_cross_track_fp(fps_track, driver_ids):
    rows = []
    for did in driver_ids:
        row = {'driver_id': did, 'n_tracks': 0}
        dim_vals = {d: [] for d in DIM_NAMES}
        for track, fp in fps_track.items():
            drv = fp[fp['driver_id'] == did]
            if len(drv) == 0:
                continue
            n_valid = 1
            if track in matrices:
                m = matrices[track]
                m_drv = m[m['driver_id'] == did]
                if len(m_drv) > 0:
                    n_valid = int(m_drv.iloc[0].get('n_corners_valid', 1))
            for d in DIM_NAMES:
                val = drv.iloc[0][d]
                if pd.notna(val):
                    dim_vals[d].append((val, n_valid))
            row['n_tracks'] += 1
        for d in DIM_NAMES:
            if dim_vals[d]:
                vals, wts = zip(*dim_vals[d])
                row[d] = float(np.average(vals, weights=wts))
            else:
                row[d] = np.nan
        rows.append(row)
    return pd.DataFrame(rows)

if fps_track:
    fp_cross = compute_cross_track_fp(fps_track, common)
    print(f"Cross-track fingerprint: {len(fp_cross)} surucu")
else:
    print("Cross-track fingerprint: uretilmedi (veri yok)")

## Faz 3: NB10A — Clustering Validation

In [ ]:
X = fp_cross[DIM_NAMES].fillna(0.5).values
n_drivers = len(fp_cross)
max_k = min(n_drivers - 1, 6)

# --- v2 KORUMA ---
# v1'de bu dalda sil_k3 tanimsiz kaliyordu ve sonraki hucreler NameError veriyordu.
sil_k3 = None
sil_agg = None
sil_fcm = None
best_sil_k = None

if n_drivers < 4:
    print(f"UYARI: Sadece {n_drivers} surucu var — kumeleme anlamsiz!")
    print("Pipeline burada duruyor (sonraki hucreler guvenli sekilde atlanacak).")
else:
    print(f"Surucu sayisi: {n_drivers}, max k: {max_k}")

    # Validity indices
    print(f"\n{'k':>3s}  {'Silhouette':>11s}  {'Cal-Har':>9s}  {'Dav-Boul':>9s}")
    print("-" * 40)

    best_sil_k = 2
    best_sil = -1

    for k in range(2, max_k + 1):
        km = KMeans(n_clusters=k, random_state=42, n_init=20)
        labels = km.fit_predict(X)
        sil = silhouette_score(X, labels)
        ch  = calinski_harabasz_score(X, labels)
        db  = davies_bouldin_score(X, labels)
        mark = " <--" if sil > best_sil else ""
        if sil > best_sil:
            best_sil = sil
            best_sil_k = k
        print(f"  {k}   {sil:11.3f}  {ch:9.2f}  {db:9.3f}{mark}")

    # k=3 ile kumeleme (T1 ile karsilastirma icin)
    km3 = KMeans(n_clusters=3, random_state=42, n_init=20)
    fp_cross['cluster'] = km3.fit_predict(X)
    sil_k3 = silhouette_score(X, fp_cross['cluster'].values)

    # Agglomerative
    if n_drivers >= 3:
        agg = AgglomerativeClustering(n_clusters=3, linkage='ward')
        labels_agg = agg.fit_predict(X)
        sil_agg = silhouette_score(X, labels_agg)
    else:
        sil_agg = -1

    # FCM
    def fuzzy_cmeans_simple(X, c, m=2.0, max_iter=300, tol=1e-6, seed=42):
        np.random.seed(seed)
        n = X.shape[0]
        U = np.random.dirichlet(np.ones(c), n)
        for _ in range(max_iter):
            Um = U ** m
            centers = (Um.T @ X) / Um.sum(axis=0)[:, np.newaxis]
            dist = np.zeros((n, c))
            for j in range(c):
                diff = X - centers[j]
                dist[:, j] = np.sqrt(np.sum(diff**2, axis=1))
            dist = np.maximum(dist, 1e-10)
            U_new = np.zeros((n, c))
            for j in range(c):
                denom = np.sum((dist[:, j:j+1] / dist) ** (2/(m-1)), axis=1)
                U_new[:, j] = 1.0 / denom
            if np.max(np.abs(U_new - U)) < tol:
                break
            U = U_new
        return np.argmax(U, axis=1)

    labels_fcm = fuzzy_cmeans_simple(X, c=3)
    sil_fcm = silhouette_score(X, labels_fcm)

    print(f"\n{'='*50}")
    print(f"  {TIER} KUMELEME OZETI (k=3)")
    print(f"{'='*50}")
    print(f"  K-Means Silhouette:       {sil_k3:.3f}")
    print(f"  Agglomerative Silhouette: {sil_agg:.3f}")
    print(f"  FCM Silhouette:           {sil_fcm:.3f}")
    print(f"  En iyi k (Silhouette):    {best_sil_k}")

    # --- v2 DEGISIKLIK 2: T1 karsilastirmasi t1_info.json'dan ---
    # v1: T1 parquet'i okunup yeniden kumeleniyordu; dosya FP_DIR kokundeydi
    #     (NB09 v3 ciktisi), yani "T1" aslinda tek-arac garantisi tasimiyordu.
    _bl = load_baseline_t1()
    if TIER == "T1":
        print(f"\n  (Bu calisma T1'in kendisi — baseline karsilastirmasi atlandi)")
    elif _bl is None:
        print(f"\n  UYARI: {T1_FP_DIR / 't1_info.json'} bulunamadi.")
        print(f"  Once TIER='T1' ile calistir, sonra bu tier'a don.")
    else:
        t1_sil = _bl['silhouette_k3']
        print(f"\n  T1 Silhouette:            {t1_sil:.3f}   (kaynak: t1_info.json)")
        print(f"  Delta ({TIER} - T1):       {sil_k3 - t1_sil:+.3f}")

## Faz 4: NB10B — SHAP + RF Validation

In [ ]:
# --- v2 KORUMA: sentinel degerler ---
acc  = None
loocv_n_correct = None
loocv_n_total   = None
rho  = None
p_val = None
top5 = None
available_metrics = []

if n_drivers < 4:
    print("Yetersiz surucu — SHAP atlaniyor")
else:
    # Metrik matrisi olustur
    available_metrics = [m for m in ALL_METRICS if any(m in matrices[t].columns for t in matrices)]

    # Cross-track raw matris
    raw_rows = []
    for did in common:
        row = {'driver_id': did}
        for m in available_metrics:
            vals = []
            for track, mat in matrices.items():
                drv = mat[mat['driver_id'] == did]
                if len(drv) > 0 and m in drv.columns and pd.notna(drv.iloc[0][m]):
                    vals.append(float(drv.iloc[0][m]))
            row[m] = np.mean(vals) if vals else np.nan
        raw_rows.append(row)

    raw_df = pd.DataFrame(raw_rows)
    X_raw = raw_df[available_metrics].fillna(0).values
    y = fp_cross['cluster'].values

    # RF + LOOCV
    rf = RandomForestClassifier(
        n_estimators=500, max_depth=4, min_samples_leaf=3,
        random_state=42, class_weight='balanced'
    )

    loo = LeaveOneOut()
    y_pred = cross_val_predict(rf, X_raw, y, cv=loo)
    acc = accuracy_score(y, y_pred)
    loocv_n_correct = int((y == y_pred).sum())
    loocv_n_total   = int(len(y))

    # SHAP
    rf.fit(X_raw, y)

    try:
        import shap
    except ImportError:
        import subprocess
        subprocess.check_call(['pip', 'install', 'shap', '-q'])
        import shap

    explainer = shap.TreeExplainer(rf)
    shap_raw = explainer.shap_values(X_raw)

    if isinstance(shap_raw, list):
        shap_list = shap_raw
    else:
        shap_arr = np.array(shap_raw)
        if shap_arr.ndim == 3:
            shap_list = [shap_arr[:, :, ci] for ci in range(shap_arr.shape[2])]
        else:
            shap_list = [shap_arr]

    mean_abs_shap = np.mean([np.abs(sv) for sv in shap_list], axis=0).mean(axis=0)
    shap_rank = {available_metrics[i]: rank+1 for rank, i in enumerate(np.argsort(mean_abs_shap)[::-1])}

    # RF Gini vs SHAP korelasyon
    gini = rf.feature_importances_
    gini_rank = {available_metrics[i]: rank+1 for rank, i in enumerate(np.argsort(gini)[::-1])}

    rf_ranks = [gini_rank[m] for m in available_metrics]
    shap_ranks = [shap_rank[m] for m in available_metrics]
    rho, p_val = spearmanr(rf_ranks, shap_ranks)

    # Top 5
    top5_idx = np.argsort(mean_abs_shap)[::-1][:5]
    top5 = [(available_metrics[i], mean_abs_shap[i]) for i in top5_idx]

    print(f"{'='*60}")
    print(f"  {TIER} SHAP + RF SONUCLARI")
    print(f"{'='*60}")
    print(f"  LOOCV Dogruluk:   {acc:.1%} ({sum(y == y_pred)}/{len(y)})")
    print(f"  RF vs SHAP rho:   {rho:.3f} (p={p_val:.4f})")
    print(f"\n  Top 5 metrik (SHAP):")
    for m, v in top5:
        print(f"    {m:<25s}  SHAP={v:.4f}")

    # --- v2 DEGISIKLIK 2 (devami): T1 satiri hardcode degil ---
    # v1: print("    T1: LOOCV=62.5%, rho=0.984, #1=coast_dist")
    _bl = load_baseline_t1()
    print(f"\n  T1 karsilastirma:")
    if TIER == "T1":
        print(f"    (Bu calisma T1'in kendisi)")
    elif _bl is None:
        print(f"    T1 referansi yok — once TIER='T1' calistir.")
    else:
        print(f"    T1: LOOCV={_bl['loocv_accuracy']:.1%}, "
              f"rho={_bl['rf_shap_rho']:.3f}, #1={_bl['top1_metric']}   (kaynak: t1_info.json)")
    print(f"    {TIER}: LOOCV={acc:.1%}, rho={rho:.3f}, #1={top5[0][0]}")

## Faz 5: Kaydet + Ozet

In [ ]:
import json
from datetime import datetime

# Kaydet
if n_drivers >= 4:
    fp_cross.to_parquet(TIER_FP / 'fingerprint_cross_track.parquet', index=False)

    for track, mat in matrices.items():
        mat.to_parquet(TIER_FEATURES / f'driver_corner_matrix_{track}.parquet', index=False)

    # Profil JSON
    profiles = {}
    for ci in range(3):
        members = fp_cross[fp_cross['cluster'] == ci]
        center = {d: float(members[d].mean()) for d in DIM_NAMES}
        profiles[str(ci)] = {
            'center': {k: round(v, 4) for k, v in center.items()},
            'members': members['driver_id'].tolist(),
            'n_members': len(members),
        }
    with open(TIER_FP / 'cluster_profiles.json', 'w', encoding='utf-8') as f:
        json.dump(profiles, f, indent=2, ensure_ascii=False)

    # Info JSON — v2: paper_numbers icin ek izlenebilirlik alanlari
    info = {
        'tier': TIER,
        'n_drivers_cross_track': len(fp_cross),
        'n_drivers_total': len(total_drivers),
        'cars': TIER_CAR_LIST,
        'tracks': list(matrices.keys()),
        'silhouette_k3': round(sil_k3, 4),
        'loocv_accuracy': round(acc, 4),
        'loocv_n_correct': loocv_n_correct,
        'loocv_n_total': loocv_n_total,
        'cross_track_driver_ids': sorted(fp_cross['driver_id'].tolist()),
        'cross_track_n_records': int(len(fp_cross)),
        'cross_track_n_persons': int(len({str(_d).split('_', 1)[1]
                                          for _d in fp_cross['driver_id'] if '_' in str(_d)})),
        'cross_track_persons': sorted({str(_d).split('_', 1)[1]
                                       for _d in fp_cross['driver_id'] if '_' in str(_d)}),
        'rf_shap_rho': round(rho, 4),
        'top1_metric': top5[0][0],
        # --- v2 EK ALANLAR ---
        'n_drivers_per_track': {t: int(m['driver_id'].nunique()) for t, m in matrices.items()},
        'degenerate_ids_excluded': DEGENERATE_IDS,
        'identity_policy': POLICY,
        'ablock_excluded': (POLICY == 'P_EXC'),
        'exclude_hard': EXCLUDE_HARD,
        'cross_track_driver_ids': list(common),
        'silhouette_best_k': int(best_sil_k),
        'silhouette_agglomerative_k3': round(float(sil_agg), 4),
        'silhouette_fcm_k3': round(float(sil_fcm), 4),
        'tier_source_dir': str(TIER_DIR),
        'notebook': 'NB11_tier_pipeline_v3_2',
        'id_unit': 'driver_id (oturum-gunu) - KISI DEGIL. NB16 person birimiyle KARISTIRMA.',
        'loocv_caveat': ('cross-track sette ayni kisinin birden fazla oturum-gunu olabilir; '
                         'leave-one-record-out kardes kayitlari egitimde birakir -> '
                         'dogruluk UST SINIR olarak okunmali. T1 temiz (tekrar yok), '
                         'T2/T3/T4 uc kardes kayit tasiyor.'),
        'run_timestamp': datetime.now().isoformat(timespec='seconds'),
    }
    with open(TIER_FP / f'{TIER.lower()}_info.json', 'w', encoding='utf-8') as f:
        json.dump(info, f, indent=2, ensure_ascii=False)

    print(f"Kaydedilen dosyalar ({TIER}):")
    print(f"  {TIER_FP / 'fingerprint_cross_track.parquet'}")
    print(f"  {TIER_FP / 'cluster_profiles.json'}")
    print(f"  {TIER_FP / f'{TIER.lower()}_info.json'}")
    for track in matrices:
        print(f"  {TIER_FEATURES / f'driver_corner_matrix_{track}.parquet'}")
else:
    print(f"n_drivers={n_drivers} < 4 — kayit yapilmadi, info JSON uretilmedi.")

# Buyuk ozet tablo
print(f"\n{'='*60}")
print(f"  {TIER} PIPELINE OZET TABLOSU")
print(f"{'='*60}")
print(f"  Tier:                 {TIER}")
print(f"  Kaynak dizin:         {TIER_DIR.name}")
print(f"  Araclar:              {TIER_CAR_LIST}")
print(f"  Pistler:              {list(matrices.keys())}")
print(f"  Toplam surucu:        {len(total_drivers)}")
print(f"  Cross-track surucu:   {len(common)}")

if n_drivers >= 4:
    print(f"  Silhouette (k=3):     {sil_k3:.3f}")
    print(f"  LOOCV Dogruluk:       {acc:.1%}")
    print(f"  RF vs SHAP rho:       {rho:.3f}")
    print(f"  Top 1 metrik:         {top5[0][0]}")

    # --- v2 DEGISIKLIK 4: t1_vals hardcode kaldirildi ---
    # v1: t1_vals = {'Silhouette': 0.351, 'LOOCV': 0.625, 'rho': 0.984}
    _bl = load_baseline_t1()
    if TIER == "T1":
        print(f"\n  (T1 baseline — karsilastirma yok)")
    elif _bl is None:
        print(f"\n  --- T1 KARSILASTIRMA: t1_info.json bulunamadi ---")
        print(f"      {T1_FP_DIR / 't1_info.json'}")
    else:
        print(f"\n  --- T1 KARSILASTIRMA (kaynak: t1_info.json) ---")
        print(f"  {'Metrik':<25s}  {'T1':>8s}  {TIER:>8s}  {'Delta':>8s}")
        print(f"  {'-'*55}")
        t1_vals   = {'Silhouette': _bl['silhouette_k3'],
                     'LOOCV':      _bl['loocv_accuracy'],
                     'rho':        _bl['rf_shap_rho']}
        tier_vals = {'Silhouette': sil_k3, 'LOOCV': acc, 'rho': rho}
        for metric in ['Silhouette', 'LOOCV', 'rho']:
            t1v, tv = t1_vals[metric], tier_vals[metric]
            print(f"  {metric:<25s}  {t1v:>8.3f}  {tv:>8.3f}  {tv - t1v:>+8.3f}")
print(f"{'='*60}")

## Ek — disk envanteri (tanisal)


In [ ]:
from pathlib import Path

P = (TRACK_ROOT / "data" / "processed")
TIERS = ["tier1_small_3t1c", "tier2_medium_3t2c", "tier3_large_3t3c", "tier4_full_4t3c"]

print(f"{'dizin':<22} {'human':>6} {'tum_pq':>8} {'toplam_MB':>10}")
print("-" * 50)
for n in TIERS:
    d = P / n
    if not d.exists():
        print(f"{n:<22}  YOK")
        continue
    human = list(d.glob("*_human.parquet"))
    allpq = list(d.rglob("*.parquet"))
    mb = sum(f.stat().st_size for f in allpq) / 1e6
    print(f"{n:<22} {len(human):>6} {len(allpq):>8} {mb:>10.1f}")

# Alt dizin var mi (stint dosyalari nerede duruyor)
print("\nAlt dizinler:")
for n in TIERS:
    d = P / n
    if not d.exists():
        continue
    subs = [x.name for x in d.iterdir() if x.is_dir()]
    print(f"  {n:<22} {subs if subs else '(alt dizin yok)'}")

In [ ]:
# ============================================================
# NB11 v3.1 YENIDEN KOSU DOGRULAMASI
# 19 Tem (v3) degerleri ile 22 Tem (v3.1) degerlerini karsilastirir.
# Taban: bu oturumda yapistirilan sekiz t*_info.json dokumu.
# ============================================================
import json, glob, os

BASELINE = {
 't1_exc': dict(n_drivers_cross_track=8,  n_drivers_total=18, silhouette_k3=0.3769,
                loocv_accuracy=0.625,  loocv_n_correct=5,  loocv_n_total=8,
                rf_shap_rho=0.9702, top1_metric='trail_braking_ratio', silhouette_best_k=2,
                silhouette_agglomerative_k3=0.3769, silhouette_fcm_k3=0.3769,
                n_drivers_per_track={'monza':10,'barcelona':17,'red_bull_ring':10}),
 't1_inc': dict(n_drivers_cross_track=10, n_drivers_total=21, silhouette_k3=0.446,
                loocv_accuracy=0.8,    loocv_n_correct=8,  loocv_n_total=10,
                rf_shap_rho=0.9789, top1_metric='apex_speed_std', silhouette_best_k=3,
                silhouette_agglomerative_k3=0.446, silhouette_fcm_k3=0.446,
                n_drivers_per_track={'monza':12,'barcelona':20,'red_bull_ring':13}),
 't2_exc': dict(n_drivers_cross_track=13, n_drivers_total=19, silhouette_k3=0.2996,
                loocv_accuracy=0.6154, loocv_n_correct=8,  loocv_n_total=13,
                rf_shap_rho=0.9772, top1_metric='mean_apex_speed', silhouette_best_k=2,
                silhouette_agglomerative_k3=0.3016, silhouette_fcm_k3=0.3263,
                n_drivers_per_track={'monza':14,'barcelona':18,'red_bull_ring':16}),
 't2_inc': dict(n_drivers_cross_track=16, n_drivers_total=22, silhouette_k3=0.374,
                loocv_accuracy=0.75,   loocv_n_correct=12, loocv_n_total=16,
                rf_shap_rho=0.9877, top1_metric='mean_apex_speed', silhouette_best_k=2,
                silhouette_agglomerative_k3=0.3598, silhouette_fcm_k3=0.374,
                n_drivers_per_track={'monza':17,'barcelona':21,'red_bull_ring':19}),
 't3_exc': dict(n_drivers_cross_track=13, n_drivers_total=21, silhouette_k3=0.2654,
                loocv_accuracy=0.6923, loocv_n_correct=9,  loocv_n_total=13,
                rf_shap_rho=0.9789, top1_metric='mean_apex_speed', silhouette_best_k=2,
                silhouette_agglomerative_k3=0.2932, silhouette_fcm_k3=0.2654,
                n_drivers_per_track={'monza':14,'barcelona':20,'red_bull_ring':16}),
 't3_inc': dict(n_drivers_cross_track=16, n_drivers_total=24, silhouette_k3=0.3407,
                loocv_accuracy=0.875,  loocv_n_correct=14, loocv_n_total=16,
                rf_shap_rho=0.9842, top1_metric='mean_apex_speed', silhouette_best_k=2,
                silhouette_agglomerative_k3=0.3631, silhouette_fcm_k3=0.3631,
                n_drivers_per_track={'monza':17,'barcelona':23,'red_bull_ring':19}),
 't4_exc': dict(n_drivers_cross_track=13, n_drivers_total=21, silhouette_k3=0.2654,
                loocv_accuracy=0.6923, loocv_n_correct=9,  loocv_n_total=13,
                rf_shap_rho=0.9789, top1_metric='mean_apex_speed', silhouette_best_k=2,
                silhouette_agglomerative_k3=0.2932, silhouette_fcm_k3=0.2654,
                n_drivers_per_track={'monza':14,'barcelona':20,'red_bull_ring':16}),
 't4_inc': dict(n_drivers_cross_track=16, n_drivers_total=24, silhouette_k3=0.3407,
                loocv_accuracy=0.875,  loocv_n_correct=14, loocv_n_total=16,
                rf_shap_rho=0.9842, top1_metric='mean_apex_speed', silhouette_best_k=2,
                silhouette_agglomerative_k3=0.3631, silhouette_fcm_k3=0.3631,
                n_drivers_per_track={'monza':17,'barcelona':23,'red_bull_ring':19}),
}

ROOT = str(TRACK_ROOT / "data" / "fingerprints")
TOL  = 1e-9          # birebir bekliyoruz; sapma varsa buyuklugunu goster

n_ok = n_diff = n_missing = 0
diffs, cross_ids, versions = [], {}, {}

for key in sorted(BASELINE):
    path = os.path.join(ROOT, key, key.split('_')[0] + '_info.json')
    if not os.path.exists(path):
        print('EKSIK DOSYA :', path); n_missing += 1; continue
    d = json.load(open(path, encoding='utf-8'))
    versions[key] = (d.get('notebook'), d.get('run_timestamp'),
                     'id_unit' in d, 'loocv_caveat' in d)
    cross_ids[key] = d.get('cross_track_driver_ids')
    row_ok = True
    for f, exp in BASELINE[key].items():
        got = d.get(f)
        if isinstance(exp, float):
            same = (got is not None) and abs(float(got) - exp) <= TOL
        else:
            same = (got == exp)
        if not same:
            row_ok = False
            diffs.append((key, f, exp, got))
    print(('OK   ' if row_ok else 'FARK ') + key)
    n_ok += row_ok; n_diff += (not row_ok)

print('\n' + '=' * 74)
print('SONUC: {} tier ayni, {} tier farkli, {} dosya eksik'.format(n_ok, n_diff, n_missing))
if diffs:
    print('\n--- FARKLAR ---')
    for k, f, e, g in diffs:
        print('  {:<8} {:<30} 19Tem={!r:<24} 22Tem={!r}'.format(k, f, e, g))

print('\n--- SURUM / ETIKET DENETIMI (v3.1 kosuldu mu) ---')
for k, (nbname, ts, has_unit, has_cav) in sorted(versions.items()):
    bayrak = '' if (nbname == 'NB11_tier_pipeline_v3_1' and has_unit and has_cav) else '   <-- v3.1 DEGIL'
    print('  {:<8} {:<28} {}  id_unit={} loocv_caveat={}{}'.format(
          k, str(nbname), ts, has_unit, has_cav, bayrak))

print('\n--- CROSS-TRACK KAYIT LISTELERI (kirpilmadan) ---')
for k in sorted(cross_ids):
    ids = cross_ids[k] or []
    persons = sorted({i.split('_', 1)[1] for i in ids if '_' in i})
    print('  {:<8} kayit={:<3} kisi={:<3} {}'.format(k, len(ids), len(persons), persons))
    print('           ', ids)

In [ ]:
import json, glob, os
for p in sorted(glob.glob(str(TRACK_ROOT / "data" / "fingerprints" / "t*_*" / "t*_info.json"))):
    d = json.load(open(p, encoding='utf-8'))
    print('{:<8} kayit={} kisi={} {}'.format(
        os.path.basename(os.path.dirname(p)),
        d.get('cross_track_n_records'), d.get('cross_track_n_persons'),
        d.get('cross_track_persons')))